# Experimento — Remoção das Variáveis Categóricas, 3 Classificações e Com Balanceamento

## Objetivo do Experimento

Após os testes anteriores utilizando:
- variáveis categóricas;
- diferentes quantidades de classificações;
- e cenários com e sem balanceamento,

foi iniciado um novo experimento removendo completamente as variáveis categóricas do modelo.

Nesse cenário, o treinamento passou a utilizar apenas variáveis físico-químicas contínuas:
- `Temperature`
- `Orthophosphate`
- `Nitrogen`

Além disso:
- o modelo continuou trabalhando com 3 classificações:
  - `Adequada`
  - `Atenção`
  - `Crítica`
- e foi aplicado balanceamento das classes durante o treinamento.

---

# O Que Será Analisado?

O principal objetivo desse experimento é entender:

- se as variáveis categóricas realmente ajudam o modelo;
- ou se os parâmetros físico-químicos já conseguem representar suficientemente os padrões ambientais.

Além disso, esse experimento também busca analisar:
- impacto do balanceamento;
- comportamento do overfitting;
- capacidade de generalização;
- comportamento das classes minoritárias;
- e principalmente o recall da classe `Crítica`.

---



# Comparações que Serão Realizadas

Os resultados desse experimento serão comparados principalmente com:
- o experimento equivalente utilizando 4 classificações.

O objetivo é analisar:
- impacto da redução para 3 classes;
- impacto da remoção das categóricas;
- influência do balanceamento;
- e comportamento geral do Random Forest em diferentes configurações.

## Preparação do ambiente


In [ ]:
# IMPORTAÇÃO DE BIBLIOTECAS
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

SEED = 42

In [ ]:
# DETECÇÃO DE AMBIENTE
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# CONFIGURAÇÃO DE CAMINHO E CARREGAMENTO
if IN_COLAB:
    print("Ambiente Google Colab detectado.")

    drive.mount('/content/drive')

    DATA_PATH = Path(
        "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/amostra_rotulada_3.parquet"
    )

else:
    print("Ambiente local/VS Code detectado.")

    DATA_PATH = Path("../../dataset/amostra_rotulada_3.parquet")

# LEITURA DO DATASET
df = pd.read_parquet(DATA_PATH)

print("Dataset Parquet carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")

df.head()

Ambiente Google Colab detectado.
Mounted at /content/drive
Dataset Parquet carregado com sucesso.
Shape do dataset: (141399, 22)


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,CCME_Values,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status
0,Canada,CZPOH_1117,River,05-03-2012,0.092790,2.25455,9.43636,0.06100,7.50000,9.40000,...,93.116725,Good,1,1,1,1,3.7,1,5,Adequada
1,Canada,FISW_32,Lake,02-12-2003,0.043792,2.13333,9.82400,0.00200,7.79000,12.00000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Adequada
2,Canada,CZPOH_1036,River,12-03-2018,0.085920,2.01667,11.78333,0.02568,7.36667,8.91667,...,100.000000,Excellent,1,1,1,1,3.7,1,5,Adequada
3,Canada,IEEA_10_32,Lake,08-06-2001,0.015920,0.55000,9.82400,0.00400,7.79000,16.80000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Adequada
4,Canada,ES0910524,River,11-09-2012,0.150000,2.13333,10.32500,0.07250,7.79000,8.32500,...,92.882835,Good,1,1,1,1,2.0,1,5,Adequada


In [ ]:
# preparando o ambiente para machine learning
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

In [ ]:
# definição de x e y
X = df[
    [
        "Temperature (cel)",
        "Orthophosphate (mg/l)",
        "Nitrogen (mg/l)"
    ]
]

y = df["conama_status"]

In [ ]:
#train_test split

SEED = 42

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)


Treino: (113119, 3)
Teste: (28280, 3)


In [ ]:
model = Pipeline(
    steps=[

       (
            "classifier",
            RandomForestClassifier(
                random_state=SEED,
                class_weight="balanced"
            )
        )
    ]
)

In [ ]:
model.fit(X_train, y_train)

Pipeline(steps=[('classifier',
                 RandomForestClassifier(class_weight='balanced',
                                        random_state=42))])

## Avaliação das Métricas de Treino

Antes de analisar os resultados do conjunto de teste, também foi realizada a avaliação das métricas no conjunto de treino.

Essa etapa é importante porque permite comparar o comportamento do modelo entre treino e teste, ajudando na identificação de possíveis sinais de overfitting.

Por esse motivo, não basta apenas observar uma alta acurácia no treino. O ideal é que os resultados de treino e teste permaneçam relativamente próximos, indicando que o modelo conseguiu aprender padrões mais generalizáveis ao invés de apenas memorizar os dados.

Além da acurácia, também foram analisadas métricas como precision, recall e F1-score, permitindo observar como o modelo se comporta em cada classe individualmente.

In [ ]:
y_train_pred = model.predict(X_train)

In [ ]:
train_accuracy = accuracy_score(y_train, y_train_pred)

print("Train Accuracy:")
print(train_accuracy)

Train Accuracy:
0.9114030357411221


In [ ]:
train_accuracy = accuracy_score(y_train, y_train_pred)

train_precision = precision_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_recall = recall_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_f1 = f1_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_confusion_matrix = confusion_matrix(
    y_train,
    y_train_pred
)

print("Train Accuracy:")
print(train_accuracy)

print("Train Precision:")
print(train_precision)

print("Train Recall:")
print(train_recall)

print("Train F1:")
print(train_f1)

print("Train Confusion Matrix:")
print(train_confusion_matrix)

Train Accuracy:
0.9114030357411221
Train Precision:
0.9225005253108584
Train Recall:
0.9114030357411221
Train F1:
0.9139553466217276
Train Confusion Matrix:
[[74795  7649   331]
 [ 1796 27311   131]
 [    5   110   991]]


In [ ]:
y_pred = model.predict(X_test)

In [ ]:

print("Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.7745756718528995

Classification Report:
              precision    recall  f1-score   support

    Adequada       0.86      0.84      0.85     20694
     Atenção       0.57      0.61      0.59      7309
     Crítica       0.05      0.03      0.04       277

    accuracy                           0.77     28280
   macro avg       0.49      0.49      0.49     28280
weighted avg       0.78      0.77      0.78     28280


Confusion Matrix:
[[17403  3210    81]
 [ 2749  4494    66]
 [   45   224     8]]


# Resultados Obtidos

## Resultados Gerais

### Treino
- Accuracy de treino: `91.14%`

### Teste
- Accuracy de teste: `77.45%`

A diferença entre treino e teste ficou em aproximadamente:
- `13.69%`

Isso mostra que ainda existe overfitting, porém em um nível relativamente controlado considerando a complexidade do problema e o desbalanceamento presente nos dados.

---

# Classification Report

| Classe | Precision | Recall | F1-Score |
|---|---|---|---|
| `Adequada` | 0.86 | 0.84 | 0.85 |
| `Atenção` | 0.57 | 0.61 | 0.59 |
| `Crítica` | 0.05 | 0.03 | 0.04 |

---

# Análise dos Resultados

Mesmo removendo completamente:
- `Country`
- `Waterbody Type`

o modelo continuou apresentando um desempenho relativamente forte utilizando apenas:
- temperatura;
- ortofosfato;
- nitrogênio.

Isso mostra que os parâmetros físico-químicos possuem uma forte capacidade de representação dos padrões ambientais.

---

# Classe `Adequada`

A classe `Adequada` apresentou:
- precision: `0.86`
- recall: `0.84`
- f1-score: `0.85`

Os resultados mostram que o modelo continua identificando muito bem as amostras adequadas mesmo sem utilizar contexto categórico.

Isso reforça que:
- os próprios parâmetros ambientais já carregam bastante informação relevante para a classificação.

---

# Classe `Atenção`

A classe `Atenção` apresentou um comportamento bastante interessante:
- precision: `0.57`
- recall: `0.61`
- f1-score: `0.59`

O recall acima de `60%` indica que o modelo conseguiu identificar boa parte das amostras intermediárias.

Isso sugere que:
- a redução para 3 classificações ajudou bastante a consolidar os padrões intermediários;
- reduzindo fragmentação entre classes.

---

# Classe `Crítica`

A classe `Crítica` continuou sendo a mais difícil de aprender.

Resultados:
- precision: `0.05`
- recall: `0.03`
- f1-score: `0.04`

Mesmo com balanceamento, o recall permaneceu baixo.

Isso indica que o principal problema continua sendo:
- a baixa quantidade de amostras críticas;
- e o forte desbalanceamento estrutural do dataset.

---

# Comparação com o Experimento de 4 Classificações

## Accuracy Geral

### 4 classificações
- Accuracy de teste: `71.02%`

### 3 classificações
- Accuracy de teste: `77.45%`

Houve uma melhora de aproximadamente:
- `+6.4%`

Essa foi uma melhora bastante significativa.

---

# Overfitting

## 4 classificações
- Treino: `87.88%`
- Teste: `71.02%`
- Diferença: `16.86%`

## 3 classificações
- Treino: `91.14%`
- Teste: `77.45%`
- Diferença: `13.69%`

Além da melhora na accuracy:
- também houve redução do overfitting.

Isso indica:
- melhor generalização;
- maior estabilidade;
- e classes mais consistentes.

---

# Impacto da Redução para 3 Classificações

A principal melhora aconteceu porque:
- as classes intermediárias ficaram menos fragmentadas;
- o modelo passou a trabalhar com fronteiras mais simples;
- e os padrões ambientais ficaram mais consistentes.

Antes, o modelo precisava separar:
- `Excelente`
- `Boa`
- `Atenção`
- `Crítica`

Agora:
- `Adequada`
- `Atenção`
- `Crítica`

Isso simplificou bastante o problema de classificação.

---

# Matriz de Confusão

A matriz de confusão mostrou:
- maior concentração de previsões corretas na diagonal principal;
- menos dispersão entre classes;
- e menor confusão entre padrões intermediários.

Isso mostra que:
- o modelo ficou mais consistente;
- e passou a generalizar melhor os padrões ambientais.

---

# Conclusão

Os resultados mostraram que:
- reduzir para 3 classificações melhorou significativamente o desempenho do modelo;
- remover as variáveis categóricas não prejudicou a performance;
- os parâmetros físico-químicos continuaram extremamente relevantes;
- e o modelo apresentou melhor generalização quando comparado ao cenário de 4 classificações.

Além disso:
- a classe `Atenção` apresentou melhora relevante;
- enquanto a classe `Crítica` continua limitada principalmente pela baixa quantidade de amostras disponíveis.